# Abstention and Margin Measurement

**Purpose:** Record every team classifier's raw per-crop confidence on the three
clips, thresholds disabled, so abstention thresholds can be chosen offline against
the recorded margins.  
**Inputs:** `data/raw/clip_*.mp4`, `data/processed/<clip>/player_detections.pkl`,
`config/default.yaml`.  
**Outputs:** `data/outputs/margin_measurement/<method>_<clip>.csv`, one row per
crop.  
**Backs:** `results/team_classification/margin_measurement/`.

The first cell pins the working directory to the repo root.

In [1]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(os.getcwd())

/home/jovyan/nba-video-analytics


## 1. Inputs

Loads the three clips and their cached player tracks; the production configuration
comes from `config/default.yaml`, with any per-clip prompt overrides.

In [2]:
import sys, pickle, cv2, yaml, pandas as pd
sys.path.insert(0, '.')

from basketball.team_classifier.classifier import (
    FashionCLIPClassifier, KMeansClassifier, EmbeddingClusteringClassifier
)

with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)
tc = cfg['team_classifier']
clips_cfg = cfg.get('clips') or {}

CLIPS = ['clip_1', 'clip_2', 'clip_3']

def load_clip(name):
    frames = []
    cap = cv2.VideoCapture(f'data/raw/{name}.mp4')
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    with open(f'data/processed/{name}/player_detections.pkl', 'rb') as f:
        tracks = pickle.load(f)
    return frames, tracks

data = {c: load_clip(c) for c in CLIPS}
for c in CLIPS:
    print(f'{c}: {len(data[c][0])} frames, {len(data[c][1])} track frames')

clip_1: 117 frames, 117 track frames
clip_2: 174 frames, 174 track frames
clip_3: 243 frames, 243 track frames


## 2. Measure raw margins

Runs the FashionCLIP, K-means and embedding-clustering classifiers with every
confidence and margin threshold at 0.0 and records each raw decision. K-means fits
are seeded inside the classifiers (random_state=42). The tokenizer FutureWarning in
the output is a known transformers notice and harmless.

In [3]:
os.makedirs('data/outputs/margin_measurement', exist_ok=True)

def build(method, clip):
    settings = clips_cfg.get(clip, {})
    if method == 'fashionclip':
        return FashionCLIPClassifier(
            team_1_description=settings.get('team_1_description', tc['team_1_description']),
            team_2_description=settings.get('team_2_description', tc['team_2_description']),
            reset_interval=tc['reset_interval'],
            crop_fraction=tc['crop_fraction'],
            confidence_threshold=0.0,      # measure raw, filter offline
        )
    if method == 'kmeans':
        return KMeansClassifier(
            crop_fraction=tc['crop_fraction'],
            fit_stride=tc['kmeans_fit_stride'],
            bg_distance_threshold=tc['kmeans_background_distance_threshold'],
            margin_threshold=0.0,
        )
    return EmbeddingClusteringClassifier(
        encoder=FashionCLIPClassifier(
            team_1_description='unused', team_2_description='unused',
            reset_interval=tc['reset_interval'],
            crop_fraction=tc['crop_fraction'],
            confidence_threshold=0.0,
        ),
        fit_stride=tc['embedding_fit_stride'],
        margin_threshold=0.0,
    )

for method in ['fashionclip', 'kmeans', 'embedding']:
    for clip in CLIPS:
        frames, tracks = data[clip]
        clf = build(method, clip)
        path = f'data/outputs/margin_measurement/{method}_{clip}.csv'
        clf.assign_teams(frames, tracks, cache_path=None,
                         record_path=path, clip_name=clip)
        print(f'{method} / {clip} -> {path}')

fashionclip / clip_1 -> data/outputs/margin_measurement/fashionclip_clip_1.csv


fashionclip / clip_2 -> data/outputs/margin_measurement/fashionclip_clip_2.csv


fashionclip / clip_3 -> data/outputs/margin_measurement/fashionclip_clip_3.csv


kmeans / clip_1 -> data/outputs/margin_measurement/kmeans_clip_1.csv


kmeans / clip_2 -> data/outputs/margin_measurement/kmeans_clip_2.csv


kmeans / clip_3 -> data/outputs/margin_measurement/kmeans_clip_3.csv


embedding / clip_1 -> data/outputs/margin_measurement/embedding_clip_1.csv


embedding / clip_2 -> data/outputs/margin_measurement/embedding_clip_2.csv


embedding / clip_3 -> data/outputs/margin_measurement/embedding_clip_3.csv


## 3. Distributions and abstention rates

Summarises each method's confidence distribution per clip and the abstention rate a
range of candidate thresholds would produce, applied offline to the recorded
confidences.

In [4]:
for method in ['fashionclip', 'kmeans', 'embedding']:
    print(f'\n=== {method} ===')
    for clip in CLIPS:
        df = pd.read_csv(f'data/outputs/margin_measurement/{method}_{clip}.csv')
        conf = df['confidence']
        unusable = (~df['crop_ok']).mean() * 100
        print(f'{clip}: n={len(df)}  unusable_crops={unusable:.1f}%  '
              f'conf min={conf.min():.3f} p05={conf.quantile(.05):.3f} '
              f'p25={conf.quantile(.25):.3f} med={conf.median():.3f} '
              f'p75={conf.quantile(.75):.3f} max={conf.max():.3f}')
        for t in [0.55, 0.60, 0.65, 0.70, 0.80]:
            print(f'    threshold {t}: would abstain on {(conf < t).mean()*100:.1f}%')


=== fashionclip ===
clip_1: n=728  unusable_crops=0.0%  conf min=0.500 p05=0.564 p25=0.818 med=0.967 p75=0.995 max=1.000
    threshold 0.55: would abstain on 3.7%
    threshold 0.6: would abstain on 7.1%
    threshold 0.65: would abstain on 11.4%
    threshold 0.7: would abstain on 14.8%
    threshold 0.8: would abstain on 22.9%
clip_2: n=1623  unusable_crops=0.0%  conf min=0.510 p05=0.704 p25=0.967 med=0.995 p75=0.999 max=1.000
    threshold 0.55: would abstain on 0.8%
    threshold 0.6: would abstain on 2.3%
    threshold 0.65: would abstain on 3.6%
    threshold 0.7: would abstain on 4.8%
    threshold 0.8: would abstain on 7.9%
clip_3: n=1887  unusable_crops=0.0%  conf min=0.509 p05=0.927 p25=0.993 med=0.998 p75=1.000 max=1.000
    threshold 0.55: would abstain on 0.5%
    threshold 0.6: would abstain on 0.6%
    threshold 0.65: would abstain on 1.2%
    threshold 0.7: would abstain on 1.5%
    threshold 0.8: would abstain on 2.2%

=== kmeans ===
clip_1: n=728  unusable_crops=0.0%

## 4. Outcome

Nine CSVs (three methods by three clips) are written to
`data/outputs/margin_measurement/`. The copies shipped with the repository are in
`results/team_classification/margin_measurement/`.